# Final Machine Learning Project - Question 2: Stock Price & Trend Prediction

**Mục tiêu:** Dự đoán xu hướng giá chứng khoán dựa trên dữ liệu lịch sử OHLCV và các chỉ báo phân tích kỹ thuật.

Notebook này đóng vai trò điều phối (Interactive Pipeline) kết nối và thực thi các module trong gói `src/`:
- **`src.utils` & `configs/config.yaml`:** Quản lý cấu hình, thiết bị (Device), cố định Random Seed.
- **`src.data_loader`:** Tải và kiểm tra dữ liệu OHLCV.
- **`src.features`:** Tính toán các chỉ báo kỹ thuật (RSI, MACD, SMA, EMA, ATR...).
- **`src.preprocessing`:** Xử lý dữ liệu thiếu, chuẩn hóa và phân chia Train / Val / Test theo chuỗi thời gian.
- **`src.dataset`:** Tạo nhãn xu hướng, áp dụng Sliding Window và đóng gói PyTorch DataLoader.
- **`src.models`:** Baseline, Traditional ML (Logistic Regression, Random Forest, XGBoost) và Deep Learning (1D-CNN, LSTM, CNN-LSTM, CNN-LSTM-Attention).
- **`src.trainer`:** Vòng lặp huấn luyện, Early Stopping, lưu mô hình tốt nhất.
- **`src.evaluator`:** Tính toán các chỉ số đánh giá (Accuracy, Precision, Recall, F1, Confusion Matrix, ROC-AUC).
- **`src.visualization`:** Vẽ Learning Curves, so sánh hiệu năng, ma trận nhầm lẫn và Attention Weights.

## 1. Thiết lập môi trường & Tự động Reload Modules

In [ ]:
# Kích hoạt tự động nạp lại module khi có thay đổi trong thư mục src/
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Thêm đường dẫn thư mục gốc dự án vào sys.path để import src
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"Thư mục gốc của dự án: {PROJECT_ROOT}")

## 2. Nạp cấu hình từ `configs/config.yaml` và Thiết lập Hệ thống (`src.utils`)

In [ ]:
# Tương tác với configs/config.yaml và src/utils.py
from src.utils import load_config, set_seed, get_device

# 1. Đọc cấu hình trung tâm
config = load_config(PROJECT_ROOT / "configs" / "config.yaml")

# 2. Cố định Random Seed đảm bảo tính tái lập (Reproducibility)
set_seed(config["project"]["seed"])

# 3. Xác định thiết bị tính toán (CUDA GPU / MPS / CPU)
device = get_device()
print(f"Đã tải cấu hình thành công! Thiết bị sử dụng: {device}")

## 3. Nạp và Kiểm tra Dữ liệu OHLCV (`src.data_loader`)

In [ ]:
# Tương tác với src/data_loader.py
from src.data_loader import load_csv_data, fetch_online_stock_data, validate_ohlcv_schema

# Đọc dữ liệu thô từ thư mục data/raw/ hoặc tải trực tuyến
raw_data_path = PROJECT_ROOT / config["data"]["raw_path"]
ticker = config["data"]["ticker"]

print(f"Tiến hành đọc dữ liệu cho mã: {ticker}...")
# df_raw = load_csv_data(raw_data_path / f"{ticker}.csv")
# validate_ohlcv_schema(df_raw)
# df_raw.head()

## 4. Trích xuất Chỉ báo Kỹ thuật (`src.features`)

In [ ]:
# Tương tác với src/features.py
from src.features import build_all_technical_features

print("Tính toán các chỉ báo kỹ thuật: SMA, EMA, MACD, RSI, Bollinger Bands, ATR...")
# df_features = build_all_technical_features(df_raw)
# df_features.tail()

## 5. Tiền xử lý, Chia tập dữ liệu & Chuẩn hóa (`src.preprocessing`)

In [ ]:
# Tương tác với src/preprocessing.py
from src.preprocessing import handle_missing_values, split_time_series_data, fit_transform_scaler, transform_scaler

print("Tiền xử lý dữ liệu và chia tập Train / Validation / Test theo thứ tự thời gian...")
# 1. Xử lý missing values
# df_clean = handle_missing_values(df_features)

# 2. Phân chia chuỗi thời gian (Tránh xáo trộn dữ liệu)
# train_df, val_df, test_df = split_time_series_data(
#     df_clean, 
#     train_ratio=config["data"]["train_ratio"], 
#     val_ratio=config["data"]["val_ratio"]
# )

# 3. Fit scaler trên tập Train và transform lên Val/Test để tránh Data Leakage
# train_scaled, scaler = fit_transform_scaler(train_df, save_path=PROJECT_ROOT / config["output"]["artifacts"]["scalers"] / "scaler.pkl")
# val_scaled = transform_scaler(val_df, scaler)
# test_scaled = transform_scaler(test_df, scaler)

## 6. Tạo Nhãn, Cửa sổ trượt (Sliding Window) & PyTorch DataLoaders (`src.dataset`)

In [ ]:
# Tương tác với src/dataset.py
from src.dataset import create_classification_labels, create_sliding_windows, get_dataloaders

window_size = config["features"]["window_size"]
batch_size = config["training"]["batch_size"]
print(f"Biến đổi dữ liệu sang dạng Sliding Window với kích thước cửa sổ T = {window_size}...")

# train_loader, val_loader, test_loader = get_dataloaders(
#     train_scaled, val_scaled, test_scaled, 
#     window_size=window_size, 
#     batch_size=batch_size
# )

## 7. Huấn luyện Mô hình Cơ sở & Traditional ML (`src.models`)

In [ ]:
# Tương tác với src/models/baseline.py và src/models/traditional.py
from src.models.baseline import MajorityClassBaseline, PersistenceBaseline
from src.models.traditional import get_traditional_models, TraditionalMLPipeline

print("1. Khởi tạo và đánh giá mô hình Baseline (Majority Class, Persistence)...")
# baseline_maj = MajorityClassBaseline()
# baseline_per = PersistenceBaseline()

print("2. Khởi tạo và huấn luyện các mô hình Machine Learning truyền thống (Logistic Regression, Random Forest, XGBoost)...")
# ml_pipeline = TraditionalMLPipeline(config["models"]["traditional"])
# ml_results = ml_pipeline.train_all(X_train_flat, y_train, X_test_flat, y_test)

## 8. Khởi tạo & Huấn luyện Mô hình Deep Learning (`src.models` & `src.trainer`)
- 1D-CNN (`src.models.cnn`)
- LSTM (`src.models.lstm`)
- Hybrid CNN-LSTM (`src.models.cnn_lstm`)
- CNN-LSTM-Attention (`src.models.cnn_lstm_attention`)

In [ ]:
# Tương tác với các mô hình Deep Learning và src/trainer.py
from src.models.cnn import StockCNN1D
from src.models.lstm import StockLSTM
from src.models.cnn_lstm import StockCNNLSTM
from src.models.cnn_lstm_attention import StockCNNLSTMAttention
from src.trainer import ModelTrainer, EarlyStopping

print("Khởi tạo kiến trúc Deep Learning và Trainer loop...")
# cnn_model = StockCNN1D(input_dim=..., seq_len=window_size).to(device)
# lstm_model = StockLSTM(input_dim=..., hidden_dim=64, num_layers=2).to(device)
# hybrid_model = StockCNNLSTM(input_dim=..., cnn_out_channels=32, lstm_hidden_dim=64).to(device)
# attention_model = StockCNNLSTMAttention(input_dim=..., cnn_out_channels=32, lstm_hidden_dim=64).to(device)

# early_stopping = EarlyStopping(patience=config["training"]["patience"], checkpoint_path="artifacts/models/best_attention.pt")
# trainer = ModelTrainer(attention_model, optimizer=..., criterion=..., device=device)
# history = trainer.fit(train_loader, val_loader, epochs=config["training"]["epochs"], early_stopping=early_stopping)

## 9. Đánh giá & So sánh Hiệu năng Giữa Tất Cả Các Mô Hình (`src.evaluator`)

In [ ]:
# Tương tác với src/evaluator.py
from src.evaluator import compute_classification_metrics, compare_all_models, save_metrics_report

print("Đánh giá hiệu năng và xuất bảng tổng hợp so sánh các mô hình...")
# metrics_table = compare_all_models(all_predictions, y_test)
# save_metrics_report(metrics_table, PROJECT_ROOT / config["output"]["results"]["metrics"] / "comparison_summary.csv")
# display(metrics_table)

## 10. Trực quan hóa Kết quả & Xuất Báo cáo (`src.visualization`)

In [ ]:
# Tương tác với src/visualization.py
from src.visualization import (
    plot_learning_curves, 
    plot_confusion_matrix, 
    plot_model_comparison_bar, 
    plot_roc_curves, 
    plot_attention_weights
)

fig_dir = PROJECT_ROOT / config["output"]["results"]["figures"]
print(f"Vẽ biểu đồ và lưu vào {fig_dir}...")

# 1. Vẽ Learning Curves
# plot_learning_curves(history, save_path=fig_dir / "learning_curve_attention.png")

# 2. Vẽ Confusion Matrix
# plot_confusion_matrix(cm, class_names=["Giảm/Đi ngang", "Tăng"], save_path=fig_dir / "confusion_matrix_attention.png")

# 3. Vẽ biểu đồ so sánh các mô hình
# plot_model_comparison_bar(metrics_table, metric_name="F1-Score", save_path=fig_dir / "model_comparison_f1.png")

# 4. Trực quan hóa Attention Weights
# plot_attention_weights(attention_weights, time_steps=..., save_path=fig_dir / "attention_weights_analysis.png")